In [5]:
import pandas as pd
import matplotlib as plt
import json
from datetime import datetime, timedelta

In [6]:
df_fall_26 = pd.read_csv('/Users/arabellepark/de-densification-2/output/fall_26.csv')
df_spring_26 = pd.read_csv('/Users/arabellepark/de-densification-2/output/spring_26.csv')



In [7]:
df_fall_26

df_spring_26


,course_identifier,dept_code,school_code,course_name,term,section,call_number,instructor,start_time,end_time,days
0,AFASG4004,AFAM,SAS,Turn the TV Off: Black Performance Studies,20261,001,15974,Johanna F Almiron,12:10,14:00,R 12:10PM-2:00PM
1,AFASC3997,AFAM,SAS,INDEPENDENT STUDY-NON AFAM,20261,001,20451,Obery Hendricks,NaN,NaN,NaN
2,AFASW3940,AFAM,SAS,SENIOR THESIS SEMINAR,20261,001,11372,Kellie Jones,14:10,16:00,T 2:10PM-4:00PM
3,AFASC3901,AFAM,SAS,INDEPENDENT STDY-UNDERGRADUATE,20261,001,10505,Frank Guridy,NaN,NaN,NaN
4,AFASC3901,AFAM,SAS,INDEPENDENT STDY-UNDERGRADUATE,20261,002,10506,Brandi Summers,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
7820,WMSTW4330,WSTB,BCOL,"SWANA Diasporas: Culture, Politics and Identit...",20261,001,766,Manijeh Moradian,10:10,12:00,R 10:10AM-12:00PM
7821,WMSTBC4325,WSTB,BCOL,EMBODIMENT AND BODILY DIFFERENCE,20261,001,610,Rebecca Jordan-Young,14:10,16:00,W 2:10PM-4:00PM
7822,WMSTW4317,WSTB,BCOL,ADVANCED TOPICS,20261,001,765,Neferti Tadiar,11:10,13:00,W 11:10AM-1:00PM
7823,WMSTW4310,WSTB,BCOL,CONTEM AMER JEWISH WOMEN'S LIT,20261,001,604,Agnieszka Legutko,16:10,18:00,T 4:10PM-6:00PM


In [11]:
# grad_pattern = r'.{4}G'
# remove = df['course.course_identifier'].str.contains(grad_pattern, regex=True, na=False)
# df_undergrad = df[~remove]
# df_undergrad

desired_codes = ['SEAS', 'SAS'] 

fall_filtered_df = df_fall_26[df_fall_26['school_code'].isin(desired_codes)]
display(fall_filtered_df.shape[0])
# display(fall_filtered_df)

fall_filtered_df = fall_filtered_df.dropna(subset=['days'])
fall_filtered_df = fall_filtered_df.dropna(subset=['end_time'])
fall_filtered_df = fall_filtered_df.dropna(subset=['start_time'])
fall_filtered_df.to_csv('fall_26_school.csv')
display(fall_filtered_df.shape[0])

# spring_filtered_df = df_spring_26[df_spring_26['school_code'].isin(desired_codes)]
# display(spring_filtered_df.shape[0])

df = fall_filtered_df.copy()


# drop rows with missing days/time
df = df.dropna(subset=['days'])
df['start_time'] = pd.to_datetime(df['start_time'], format='%H:%M')
df['end_time'] = pd.to_datetime(df['end_time'], format='%H:%M')

df['days'] = df['days'].str.split().str[0]


day_map = {
    'M': 'Monday',
    'T': 'Tuesday',
    'W': 'Wednesday',
    'R': 'Thursday',
    'F': 'Friday'
}
df['days'] = df['days'].fillna('').astype(str)

df['days'] = df['days'].apply(
    lambda x: [day_map.get(d) for d in x if d in day_map]
)
df = df.explode('days').reset_index(drop=True)

time_bins = pd.date_range("08:00", "20:00", freq="30min").time

rows = []

for _, row in df.iterrows():
    day = row['days']
    start = row['start_time'].time()
    end = row['end_time'].time()

    for t in time_bins:
        # convert time → minutes since midnight
        t_minutes = t.hour * 60 + t.minute
        bin_start_minutes = t_minutes
        bin_end_minutes = t_minutes + 30

        start_minutes = start.hour * 60 + start.minute
        end_minutes = end.hour * 60 + end.minute

        # overlap condition
        if start_minutes < bin_end_minutes and end_minutes > bin_start_minutes:
            rows.append({
                'Day': day,
                'Time': t
            })
binned_df = pd.DataFrame(rows)

result = (
    binned_df
    .groupby(['Day', 'Time'])
    .size()
    .reset_index(name='Share')
)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

result['Day'] = pd.Categorical(result['Day'], categories=day_order, ordered=True)
result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time

result = result.sort_values(['Day', 'Time'])

result.to_csv('fall_26.csv', index=False)

display(result)


2630

1911

/var/folders/zl/l01js9_527d_nstkmhybfn0c0000gn/T/ipykernel_26608/1935519043.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time


,Day,Time,Share
25,Monday,08:00:00,11
26,Monday,08:30:00,49
27,Monday,09:00:00,57
28,Monday,09:30:00,57
29,Monday,10:00:00,115
...,...,...,...
20,Friday,18:00:00,7
21,Friday,18:30:00,7
22,Friday,19:00:00,3
23,Friday,19:30:00,4


In [12]:
spring_filtered_df = df_spring_26[df_spring_26['school_code'].isin(desired_codes)]
display(spring_filtered_df.shape[0])
# display(fall_filtered_df)
# 

spring_filtered_df = spring_filtered_df.dropna(subset=['days'])
spring_filtered_df = spring_filtered_df.dropna(subset=['end_time'])
spring_filtered_df = spring_filtered_df.dropna(subset=['start_time'])

spring_filtered_df.to_csv('spring_26_school.csv')
display(spring_filtered_df)


df = spring_filtered_df.copy()



# drop rows with missing days/time
df = df.dropna(subset=['days'])
df['start_time'] = pd.to_datetime(df['start_time'], format='%H:%M')
df['end_time'] = pd.to_datetime(df['end_time'], format='%H:%M')

df['days'] = df['days'].str.split().str[0]


day_map = {
    'M': 'Monday',
    'T': 'Tuesday',
    'W': 'Wednesday',
    'R': 'Thursday',
    'F': 'Friday'
}
df['days'] = df['days'].fillna('').astype(str)

df['days'] = df['days'].apply(
    lambda x: [day_map.get(d) for d in x if d in day_map]
)
df = df.explode('days').reset_index(drop=True)

time_bins = pd.date_range("08:00", "20:00", freq="30min").time

rows = []

for _, row in df.iterrows():
    day = row['days']
    start = row['start_time'].time()
    end = row['end_time'].time()

    for t in time_bins:
        # convert time → minutes since midnight
        t_minutes = t.hour * 60 + t.minute
        bin_start_minutes = t_minutes
        bin_end_minutes = t_minutes + 30

        start_minutes = start.hour * 60 + start.minute
        end_minutes = end.hour * 60 + end.minute

        # overlap condition
        if start_minutes < bin_end_minutes and end_minutes > bin_start_minutes:
            rows.append({
                'Day': day,
                'Time': t
            })

binned_df = pd.DataFrame(rows)
binned_df.to_csv("filtered_spring.csv")
result = (
    binned_df
    .groupby(['Day', 'Time'])
    .size()
    .reset_index(name='Share')
)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

result['Day'] = pd.Categorical(result['Day'], categories=day_order, ordered=True)
result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time

result = result.sort_values(['Day', 'Time'])

result.to_csv('spring_26.csv', index=False)

display(result)


3700

,course_identifier,dept_code,school_code,course_name,term,section,call_number,instructor,start_time,end_time,days
0,AFASG4004,AFAM,SAS,Turn the TV Off: Black Performance Studies,20261,001,15974,Johanna F Almiron,12:10,14:00,R 12:10PM-2:00PM
2,AFASW3940,AFAM,SAS,SENIOR THESIS SEMINAR,20261,001,11372,Kellie Jones,14:10,16:00,T 2:10PM-4:00PM
14,AFASC3011,AFAM,SAS,Spirit of Justice,20261,001,17123,Nyle Fort,10:10,12:00,M 10:10AM-12:00PM
15,AFASW3004,AFAM,SAS,Introduction to Black Geographies,20261,001,10496,Brandi Summers,14:10,16:00,W 2:10PM-4:00PM
16,AFASC3001,AFAM,SAS,SING A BLACK GIRL'S SONG: THE NON FICTION WRI...,20261,001,10540,Edwidge Danticat,14:10,16:00,R 2:10PM-4:00PM
...,...,...,...,...,...,...,...,...,...,...,...
7420,STATC3702,STAT,SAS,Exploratory Data Analysis & Visualization,20261,001,17211,Joyce T Robbins,14:40,15:55,MW 2:40PM-3:55PM
7421,STATW3293,STAT,SAS,Topics in Modern Statistics,20261,001,14016,Joyce T Robbins,14:40,15:55,TR 2:40PM-3:55PM
7423,STATW1001,STAT,SAS,INTRO TO STATISTICAL REASONING,20261,001,13892,Victor H De La Pena,14:40,15:55,MW 2:40PM-3:55PM
7424,STATW1001,STAT,SAS,INTRO TO STATISTICAL REASONING,20261,002,13893,Anthony Donoghue,10:10,11:25,MW 10:10AM-11:25AM


/var/folders/zl/l01js9_527d_nstkmhybfn0c0000gn/T/ipykernel_26608/2121858432.py:77: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time


,Day,Time,Share
22,Monday,08:00:00,10
23,Monday,08:30:00,54
24,Monday,09:00:00,62
25,Monday,09:30:00,63
26,Monday,10:00:00,145
...,...,...,...
17,Friday,16:30:00,19
18,Friday,17:00:00,11
19,Friday,17:30:00,7
20,Friday,18:00:00,6


In [23]:
# fall_filtered_df

# spring_filtered_df
prefixes = ('10', '11', '12', '13')

#Fall
time_counts = fall_filtered_df['start_time'].value_counts().reset_index()
percentages = fall_filtered_df['start_time'].value_counts(normalize=True) * 100
time_counts.columns = ['StartTime', 'Frequency']
percentages.columns = ['StartTime', 'Percentage']
print(time_counts)

import datetime as dt
time_buckets = ['8 am - 10 am', '10 am - 12 pm', '12 pm - 2 pm', '2 pm - 4 pm', '4 pm - 6 pm', '6 pm - 8 pm', '8 pm - 10 pm']
fall_filtered_df['start_time'] = pd.to_datetime(fall_filtered_df['start_time'], format='%H:%M').dt.time
bins = [
    dt.time(0, 0),   # 00:00
    dt.time(0, 2),   # 02:00
    dt.time(8, 0),   # 08:00
    dt.time(10, 0),  # 10:00
    dt.time(14, 0),  # 14:00 (2 PM)
    dt.time(16, 0),  # 16:00 (4 PM)
    dt.time(22, 0),  # 22:00 (10 PM)
    dt.time(23, 59, 59) 
]
# creating a column that categorizes the start times as the sole hour value
fall_filtered_df['start_hour'] = pd.to_datetime(fall_filtered_df['start_time'], format='%H:%M').dt.hour
hour_bins = [8, 10, 12, 14, 16, 18, 20, 22]
hour_labels = ['8 am - 10 am', '10 am - 12 pm', '12 pm - 2 pm', '2 pm - 4 pm', '4 pm - 6 pm', '6 pm - 8 pm', '8 pm - 10 pm']
fall_filtered_df['time_bucket'] = pd.cut(fall_filtered_df['start_hour'], bins=hour_bins, labels=hour_labels, right=False, include_lowest=True)

percentages = fall_filtered_df['time_bucket'].value_counts(normalize=True) * 100
print(percentages)

   StartTime  Frequency
0   16:10:00        369
1   10:10:00        299
2   14:10:00        205
3   13:10:00        188
4   14:40:00        130
5   12:10:00        122
6   18:10:00        120
7   11:40:00         94
8   08:40:00         57
9   17:40:00         32
10  17:10:00         30
11  15:10:00         28
12  19:00:00         24
13  08:10:00         24
14  10:00:00         21
15  13:00:00         20
16  09:00:00         17
17  11:10:00         11
18  11:00:00         10
19  08:50:00          9
20  09:10:00          9
21  19:10:00          9
22  15:00:00          8
23  20:10:00          7
24  08:00:00          7
25  18:00:00          6
26  09:30:00          6
27  19:40:00          6
28  19:30:00          5
29  12:00:00          4
30  10:30:00          4
31  12:05:00          3
32  16:00:00          3
33  12:30:00          3
34  18:30:00          2
35  16:50:00          2
36  19:35:00          2
37  14:00:00          2
38  09:05:00          2
39  12:45:00          2
40  15:05:00    

ValueError: unconverted data remains when parsing with format "%H:%M": ":00", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

# I didn't touch anything below this

In [ ]:
# from IPython.display import FileLink
# filtered_df.to_csv('SEAS_FAS_only.csv', index=False)
# FileLink('SEAS_FAS_only.csv')

/Users/arabellepark/de-densification-2/SEAS_FAS_only.csv

In [1]:
prefixes = ('10', '11', '12', '13')
# df['contains'] = df['start_time'].str.startswith(prefixes)
# df

In [ ]:
department_summary = df.groupby('department')['contains'].agg(
        total_types='count',
        matching_types='sum'
    )
print(department_summary)


NameError: name 'df' is not defined

In [19]:
department_summary['percentage_matching'] = (department_summary['matching_types'] / department_summary['total_types']) * 100

In [21]:
top_20_departments = department_summary.sort_values(by='total_types', ascending=False).head(20)
top_20_departments_2 = department_summary.nlargest(20, 'total_types')

In [23]:
print(top_20_departments)
print(top_20_departments_2)

            total_types  matching_types  percentage_matching
department                                                  
CORE                267             101            37.827715
SOCW                218              31            14.220183
INTA                207              97            46.859903
ARPL                140              71            50.714286
PHYS                130              37            28.461538
FINC                126              14            11.111111
CHEM                111              41            36.936937
EALC                109              62            56.880734
MELC                 93              46            49.462366
ERM                  85               1             1.176471
ECON                 85              31            36.470588
STAT                 80              34            42.500000
COMS                 80              34            42.500000
DROM                 72              16            22.222222
THEA                 69 

In [25]:
df_fall = pd.read_csv('Downloads/all_classes-fall25.csv')
df_fall

,course_identifier,course_name,subject_code,subject_name,school_code,school_name,section,call_number,instructors,enrollment_count,enrollment_max,enrollment_status,days_times,mil_time_from,mil_time_to
0,CHEMW1500,GENERAL CHEMISTRY LABORATORY,CHEM,NaN,SAS,NaN,001,10482,Sarah J Hansen,24.0,46.0,O,T 1:10PM-4:50PM,13:10,16:50
1,CHEMW1500,GENERAL CHEMISTRY LABORATORY,CHEM,NaN,SAS,NaN,002,10483,Joseph C Ulichny,52.0,48.0,F,T 6:10PM-9:50PM,18:10,21:50
2,CHEMW1500,GENERAL CHEMISTRY LABORATORY,CHEM,NaN,SAS,NaN,003,10484,Joseph C Ulichny,51.0,48.0,F,W 1:10PM-4:50PM,13:10,16:50
3,CHEMW1500,GENERAL CHEMISTRY LABORATORY,CHEM,NaN,SAS,NaN,004,10485,Sarah J Hansen,22.0,46.0,O,R 1:10PM-4:50PM,13:10,16:50
4,CHEMW1411,GENERAL CHEMISTRY I - REC,CHEM,NaN,SAS,NaN,001,10521,Robert Beer,20.0,20.0,F,M 4:10PM-5:00PM,16:10,17:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8035,SUSCK5040,SUSTAINABILITY IN THE FACE OF NATURAL DISASTERS,SUSC,NaN,SPEC,NaN,001,12439,"Einat Lev, Chia-Ying Lee",9.0,20.0,O,R 4:10PM-6:00PM,16:10,18:00
8036,SUSCK5010,"CLIMATE SCI FOR DEC MAK: MOD, ANL & AP",SUSC,NaN,SPEC,NaN,001,12336,Yutian Wu,4.0,24.0,O,W 4:10PM-6:00PM,16:10,18:00
8037,SUSCK5999,CAPSTONE WORKSHOP IN SUSTAINABLE SCIENCE,SUSC,NaN,SPEC,NaN,001,12425,"Joseph Paul, Steven N Chillrud",13.0,15.0,O,T 6:10PM-8:00PM,18:10,20:00
8038,SUSCK5995,Internship In SUSC,SUSC,NaN,SPEC,NaN,D01,17079,Karen Acampado,1.0,40.0,O,NaN,NaN,NaN


In [27]:
df_spring = pd.read_csv('Downloads/all_classes-spring26.csv')
df_spring

,course_identifier,course_name,subject_code,subject_name,school_code,school_name,section,call_number,instructors,enrollment_count,enrollment_max,enrollment_status,days_times,mil_time_from,mil_time_to
0,LCRSG8450,PERSPECTVE ON LAT AMER STUDIES,LCRS,NaN,SAS,NaN,001,16449,Gustavo S Azenha,NaN,5.0,O,NaN,NaN,NaN
1,LCRSG6401,LIT/RES-LAT AMER/CARIB STUD II,LCRS,NaN,SAS,NaN,001,16450,Gustavo S Azenha,NaN,5.0,O,W 2:10PM-4:00PM,14:10,16:00
2,LAW,EXTERNSHIP: UNITED NATIONS,LAW,NaN,SLAW,NaN,001,10102,Akshaya Kumar,NaN,999.0,B,NaN,NaN,NaN
3,LAW,EXTERNSHIP: UNITED NATIONS,LAW,NaN,SLAW,NaN,002,10104,Akshaya Kumar,NaN,999.0,B,NaN,NaN,NaN
4,LAW,EXTERNSHIP: UNITED NATIONS,LAW,NaN,SLAW,NaN,003,10103,Jehanne E Henry,NaN,999.0,B,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6322,DVGOU7350,Advanced Economic Development for Internationa...,DVGO,NaN,SIPA,NaN,R02,15587,NaN,NaN,NaN,F,M 3:10PM-5:00PM,15:10,17:00
6323,DVGOU7350,Advanced Economic Development for Internationa...,DVGO,NaN,SIPA,NaN,001,10186,Eric A Verhoogen,29.0,50.0,O,T 1:10PM-3:00PM,13:10,15:00
6324,DVGOU7350,Advanced Economic Development for Internationa...,DVGO,NaN,SIPA,NaN,002,10187,Eric A Verhoogen,31.0,50.0,O,T 3:10PM-5:00PM,15:10,17:00
6325,DVGOU7325,"Inequality, Poverty, and Politics",DVGO,NaN,SIPA,NaN,001,15597,Cesar Zucco,13.0,25.0,O,T 1:10PM-3:00PM,13:10,15:00


In [29]:
course_prefixes = ['MATH', 'POLS', 'ECON', 'PSYC', 'ENGL','HIST']
filtered_df_fall = df_fall[df_fall.course_identifier.str.startswith(tuple(course_prefixes))]
filtered_df_fall['course_identifier']

1214     ECONW3413
1215     ECONW3412
1216     ECONW3412
1217     ECONW3412
1219     ECONG9016
           ...    
7150    HISTBC2695
7151    HISTBC2695
7152    HISTBC2549
7153    HISTBC2413
7154    HISTBC1062
Name: course_identifier, Length: 673, dtype: object

In [31]:
filtered_df_fall.to_csv('filtered_fall_classes.csv', index=False)

In [33]:
course_prefixes = ['MATH', 'POLS', 'ECON', 'PSYC', 'ENGL']
filtered_df_spring = df_spring[df_spring.course_identifier.str.startswith(tuple(course_prefixes))]
filtered_df_spring
filtered_df_spring.to_csv('filtered_spring_classes.csv', index=False)

In [45]:
# analysis of full or over enrolled classes
df_full_spring26 = pd.read_csv('Downloads/spring-26-al.csv')
df_full_spring26

filtered_df_fullspring26 = df_full_spring26[df_full_spring26.enrollment_status.str.startswith(tuple('F'))]
filtered_df_fullspring26

,course_identifier,course_name,subject_code,subject_name,school_code,school_name,section,call_number,instructors,enrollment_count,enrollment_max,enrollment_status,days_times,mil_time_from,mil_time_to
1,AFASG4004,Turn the TV Off: Black Performance Studies,AFAS,NaN,SAS,NaN,001,15974,Johanna F Almiron,17.0,17.0,F,R 12:10PM-2:00PM,12:10,14:00
9,AFASC3011,Spirit of Justice,AFAS,NaN,SAS,NaN,001,17123,Nyle Fort,18.0,18.0,F,M 10:10AM-12:00PM,10:10,12:00
10,AFASW3004,Introduction to Black Geographies,AFAS,NaN,SAS,NaN,001,10496,Brandi Summers,15.0,15.0,F,W 2:10PM-4:00PM,14:10,16:00
12,AFASW1003,Blackness and Frenchness: A Radical Genealogy,AFAS,NaN,SAS,NaN,001,10536,Veronique Charles,18.0,18.0,F,M 12:10PM-2:00PM,12:10,14:00
23,AFASG4080,TOPICS IN THE BLACK EXPERIENCE,AFAS,NaN,SAS,NaN,005,13264,Erica Hunt,1.0,1.0,F,R 4:10PM-6:00PM,16:10,18:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6946,WRITW1100,BEGINNING FICTION WORKSHOP,WRIT,NaN,SART,NaN,005,12807,Haley Garvin,15.0,15.0,F,W 2:10PM-4:00PM,14:10,16:00
6953,WMSTBC3138,AFFECT AND ACTIVISM,WMST,NaN,BCOL,NaN,001,761,Manijeh Moradian,16.0,16.0,F,T 2:10PM-4:00PM,14:10,16:00
6958,WMSTV4905,Labor and Life: Critiques of Capitalism,WMST,NaN,BCOL,NaN,001,767,Neferti Tadiar,16.0,16.0,F,R 12:10PM-2:00PM,12:10,14:00
6959,WMSTW4330,"SWANA Diasporas: Culture, Politics and Identit...",WMST,NaN,BCOL,NaN,001,766,Manijeh Moradian,14.0,14.0,F,R 10:10AM-12:00PM,10:10,12:00


In [36]:
time_counts = filtered_df_fullspring26['mil_time_from'].value_counts().reset_index()
percentages = filtered_df_fullspring26['mil_time_from'].value_counts(normalize=True) * 100
time_counts.columns = ['StartTime', 'Frequency']
percentages.columns = ['StartTime', 'Percentage']
print(time_counts)

NameError: name 'filtered_df_fullspring26' is not defined

In [88]:
import datetime as dt
time_buckets = ['8 am - 10 am', '10 am - 12 pm', '12 pm - 2 pm', '2 pm - 4 pm', '4 pm - 6 pm', '6 pm - 8 pm', '8 pm - 10 pm']
filtered_df_fullspring26['start_time'] = pd.to_datetime(filtered_df_fullspring26['mil_time_from'], format='%H:%M').dt.time
bins = [
    dt.time(0, 0),   # 00:00
    dt.time(0, 2),   # 02:00
    dt.time(8, 0),   # 08:00
    dt.time(10, 0),  # 10:00
    dt.time(14, 0),  # 14:00 (2 PM)
    dt.time(16, 0),  # 16:00 (4 PM)
    dt.time(22, 0),  # 22:00 (10 PM)
    dt.time(23, 59, 59) 
]
# creating a column that categorizes the start times as the sole hour value
filtered_df_fullspring26['start_hour'] = pd.to_datetime(filtered_df_fullspring26['mil_time_from'], format='%H:%M').dt.hour
hour_bins = [8, 10, 12, 14, 16, 18, 20, 22]
hour_labels = ['8 am - 10 am', '10 am - 12 pm', '12 pm - 2 pm', '2 pm - 4 pm', '4 pm - 6 pm', '6 pm - 8 pm', '8 pm - 10 pm']
filtered_df_fullspring26['time_bucket'] = pd.cut(filtered_df_fullspring26['start_hour'], bins=hour_bins, labels=hour_labels, right=False, include_lowest=True)

percentages = filtered_df_fullspring26['time_bucket'].value_counts(normalize=True) * 100
print(percentages)

NameError: name 'filtered_df_fullspring26' is not defined